# World Cup 2026 — data explorer

A **read-only** playground for the DuckDB at `data/processed/worldcup.duckdb`.

- Every query opens its own short-lived read-only connection and closes it, so this notebook **never holds the DB lock** — you can run `ingest_*.py --apply` in a terminal while this is open.
- Run cells top-to-bottom once, then poke around freely. Edit the SQL, re-run, break things — it's read-only, you can't hurt the data.
- New tables (`wc2026_squad`, `ea_fc26_player`, …) show up here automatically once their loaders' `--apply` has run.

In [ ]:
import duckdb, pandas as pd
from pathlib import Path
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Find the DB by walking up from wherever Jupyter was launched.
DB_PATH = next(p / 'data/processed/worldcup.duckdb'
               for p in [Path.cwd(), *Path.cwd().parents]
               if (p / 'data/processed/worldcup.duckdb').exists())

def q(sql: str) -> pd.DataFrame:
    """Run SQL read-only and return a DataFrame. Opens + closes a fresh
    connection each call so we never hold a lock between cells."""
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        return con.sql(sql).df()
    finally:
        con.close()

print('DB:', DB_PATH)

## 1. What tables do we have, and how big?

In [ ]:
tables = q("""SELECT table_name FROM information_schema.tables
              WHERE table_schema='main' ORDER BY table_name""")['table_name'].tolist()
counts = [(t, q(f'SELECT COUNT(*) AS n FROM "{t}"')['n'][0]) for t in tables]
pd.DataFrame(counts, columns=['table', 'rows']).sort_values('rows', ascending=False).reset_index(drop=True)

## 2. Inspect any table's columns
Change the name and re-run.

In [ ]:
def schema(t: str) -> pd.DataFrame:
    return q(f"""SELECT column_name, data_type FROM information_schema.columns
                WHERE table_name='{t}' ORDER BY ordinal_position""")

schema('player_season_stats')

In [ ]:
# peek at the actual rows
q('SELECT * FROM player_season_stats LIMIT 5')

## 3. Coverage — what do we actually know?
The project's central idea: how much data backs each player / nation.

In [ ]:
# dob is only populated for the FBref/UCL subset (~20%)
q('SELECT COUNT(*) AS players, COUNT(player_dob) AS with_dob FROM players')

In [ ]:
# which competitions/seasons do we hold, and how many team-matches each?
q("""SELECT league, season, COUNT(*) AS team_matches
      FROM team_match_stats GROUP BY league, season
      UNION ALL
      SELECT league, season, COUNT(*) FROM team_match_fbref GROUP BY league, season
      ORDER BY league, season""")

## 4. A worked join — best 2024-25 attackers by the model's form score
Shows how `players` (the dimension) joins to `player_season_stats` (the facts).

In [ ]:
q("""
SELECT p.player_name, s.team, s.league, s.minutes, s.goals, s.assists,
       ROUND(s.np_xg, 1) AS np_xg, ROUND(s.xa, 1) AS xa,
       ROUND(s.shrunk_form_eb, 3) AS form
FROM player_season_stats s
JOIN players p USING (player_id)
WHERE s.season = '2024-2025' AND s.minutes > 1500
  AND s.primary_position_class_v103 = 'FWD'
ORDER BY s.shrunk_form_eb DESC
LIMIT 15
""")

## 5. Your turn 🐢
Write any SQL below. Some ideas: team strength (`team_season_strength_v103`),
UCL players (`player_match_fbref`), or — once you've run the squad/EA applies —
join `wc2026_squad` to `ea_fc26_player` to see attribute coverage per nation.

In [ ]:
q("""
-- your SQL here
SELECT team, season, avg_xg_for, avg_xg_allowed, avg_ppda_pressing
FROM team_season_strength_v103
ORDER BY avg_xg_for DESC
LIMIT 10
""")